In [1]:
import xarray as xr
import numpy as np

## helper function

In [2]:
def help_adding_year(yr):
    """
    Update year string by adding one and return string.
    Args:
        yr (str): year
    """
    assert len(yr) == 4, 'year string is not correct length!'
    newyr = int(yr) + 1
    newyr = str(newyr)
    newyr = newyr.zfill(4)
    return newyr


def convert_mo_str(mo):
    """
    Convert integer month to two-digit month string.
    Args:
        mo (int): month
    """
    assert type(mo) is int, 'not an integer'
    return str(mo).zfill(2)

def help_updating_time(mo, yr):
    """
    Updating the month and year in cftime due to february start in cesm.
    Year and month (if december) are fixed.
    Args:
        mo (str): xarray filename
        yr (str): xarray filename
    """
    newmo = int(mo) + 1
    if newmo == 13:
        yr = help_adding_year(yr)
        newmo = 1
    assert newmo <= 12, 'month is greater than 12, there is an error!'
    newmo = convert_mo_str(newmo)
    return newmo, yr

def fixtime(ds):
    """
    Open file and add time coordinate.
    Args:
        ds (xarray dataset)
    """
    # start date range preprocessing
    mo0 = ds.encoding['source'].split('/')[-1].split('.')[-2][4:6]
    yr0 = ds.encoding['source'].split('/')[-1].split('.')[-2][:4]
    nm0, ny0 = help_updating_time(mo0, yr0)
    # end date range preprocessing
    yr1 = ds.encoding['source'].split('/')[-1].split('.')[-2][-6:-2]
    mo1 = ds.encoding['source'].split('/')[-1].split('.')[-2][-2:]
    nm1, ny1 = help_updating_time(mo1, yr1)
    # create datetime array in cftime
    newtime = xr.cftime_range(
        start=ny0+'-'+nm0+'-01',
        end=ny1+'-'+nm1+'-01',
        freq='MS',
        calendar='noleap'
    )
    ds = ds.assign_coords(time=newtime)
    return ds

## dask

In [3]:
def get_ClusterClient():
    import dask
    from dask_jobqueue import PBSCluster
    from dask.distributed import Client
    cluster = PBSCluster(
        cores=1,
        memory='10GB',
        processes=1,
        queue='casper',
        resource_spec='select=1:ncpus=1:mem=10GB',
        account='UMCP0030',
        walltime='02:00:00')
        #interface='ib0',)

    dask.config.set({
        'distributed.dashboard.link':
        'https://jupyterhub.hpc.ucar.edu/stable/user/{USER}/proxy/{port}/status'
    })
    client = Client(cluster)
    return cluster, client

In [4]:
cluster, client = get_ClusterClient()
cluster.adapt(minimum=0, maximum=10)

In [5]:
cluster

Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/ewisinski/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.101:43115,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/ewisinski/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [ ]:
cluster.close()
client.close()

## pressure data

In [8]:
ds_press = xr.open_mfdataset(
    '/glade/campaign/cgd/ccr/E3SMv2/FV_regridded/v2.FV1.piControl/atm/proc/tseries/month_1/v2.FV1.piControl.eam.h0.PS.*.nc',
    preprocess=fixtime,
    concat_dim='time',
    combine='nested'
)

In [9]:
ds_pressure = ds_press.assign_coords(time=ds_press['time'] - 
                                        xr.coding.cftime_offsets.MonthBegin(1))

In [10]:
ds_pressure = ds_pressure['PS']

In [11]:
ds_pressure

<xarray.DataArray 'PS' (time: 6000, lat: 192, lon: 288)> Size: 1GB
dask.array<concatenate, shape=(6000, 192, 288), dtype=float32, chunksize=(1, 192, 288), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * time     (time) object 48kB 0001-01-01 00:00:00 ... 0500-12-01 00:00:00
Attributes:
    units:          Pa
    long_name:      Surface pressure
    standard_name:  surface_air_pressure
    cell_methods:   time: mean
    cell_measures:  area: area

In [12]:
# Coordinates of Tahiti and Darwin
tahiti_coords = {'lat': -17.65, 'lon': -149.42}   # Tahiti
darwin_coords = {'lat': -12.46, 'lon': 130.84}   # Darwin

# Select nearest grid points
psl_tahiti = ds_pressure.sel(lat=tahiti_coords['lat'], lon=tahiti_coords['lon'], method='nearest')
psl_darwin = ds_pressure.sel(lat=darwin_coords['lat'], lon=darwin_coords['lon'], method='nearest')

# Compute SOI: standardized Tahiti minus Darwin pressure anomalies
# First, remove the mean to get anomalies
climo_tahiti = psl_tahiti.groupby('time.month').mean(dim='time', keep_attrs=True)
climo_darwin = psl_darwin.groupby('time.month').mean(dim='time', keep_attrs=True)

anom_tahiti = psl_tahiti.groupby('time.month') - climo_tahiti
anom_darwin = psl_darwin.groupby('time.month') - climo_darwin

# SOI
soi = (anom_tahiti - anom_darwin) / np.std(anom_tahiti - anom_darwin)

# soi is now an xarray.DataArray with time dimension
print(soi)

<xarray.DataArray 'PS' (time: 6000)> Size: 24kB
dask.array<truediv, shape=(6000,), dtype=float32, chunksize=(1,), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) object 48kB 0001-01-01 00:00:00 ... 0500-12-01 00:00:00
    month    (time) int64 48kB 1 2 3 4 5 6 7 8 9 10 ... 3 4 5 6 7 8 9 10 11 12


/glade/work/ewisinski/conda-envs/oldkeras/lib/python3.11/site-packages/xarray/core/indexing.py:1621: PerformanceWarning: Slicing with an out-of-order index is generating 500 times more chunks
  return self.array[key]
/glade/work/ewisinski/conda-envs/oldkeras/lib/python3.11/site-packages/xarray/core/indexing.py:1621: PerformanceWarning: Slicing with an out-of-order index is generating 500 times more chunks
  return self.array[key]


In [13]:
soi

<xarray.DataArray 'PS' (time: 6000)> Size: 24kB
dask.array<truediv, shape=(6000,), dtype=float32, chunksize=(1,), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) object 48kB 0001-01-01 00:00:00 ... 0500-12-01 00:00:00
    month    (time) int64 48kB 1 2 3 4 5 6 7 8 9 10 ... 3 4 5 6 7 8 9 10 11 12

In [14]:
soi.to_netcdf('soi_index.nc')